# DATALUS Generative Studio: Scenario Simulation & applied GenAI

This notebook demonstrates the **DATALUS Generative Studio**, focusing on applied Generative AI tasks using a pre-trained diffusion model. 
We explore how Data Scientists and Public Policy Makers can leverage synthetic data for scenario simulation, 
minority class balancing, tabular inpainting, and advanced data augmentation.

## 1. Environment Setup & Artifact Verification
Detecting environment and verifying pre-requisites. If training artifacts are missing, we run a lightning training pass.

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import numpy as np

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_studio'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_studio'
    return './datalus_studio'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

INSTALLED = Path(WORKING_DIR) / '.installed'

if not INSTALLED.exists():
    if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        !git clone --depth 1 https://github.com/emanuellcs/datalus.git "{WORKING_DIR}/source" 2>/dev/null || true
        %cd "{WORKING_DIR}/source"
        !pip install --quiet -e '.[dev]' pysus polars matplotlib seaborn nest_asyncio 2>&1 | tail -5
        sys.path.insert(0, f'{WORKING_DIR}/source')
        INSTALLED.touch()
        print()
        print('=' * 60)
        print('  INSTALLATION COMPLETE')
        print('  Click "RESTART RUNTIME" in the warning above,')
        print('  or go to Runtime > Restart and run all')
        print('=' * 60)
        os._exit(0)
    else:
        INSTALLED.touch()
        print('Local environment. Dependencies assumed installed.')
else:
    src = Path(WORKING_DIR) / 'source'
    if src.is_dir():
        sys.path.insert(0, str(src))
    print('Dependencies already installed.')

checkpoint_path = Path(f'{WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt')
if not checkpoint_path.exists():
    print('Training artifacts not found. Running lightning training pass...')
    from pysus import sih, sim, sinan
    from datetime import datetime
    import urllib.request
    import asyncio

    STATES = ['SP', 'RJ', 'MG', 'PR', 'RS', 'BA', 'DF']

    def fmt_tier(label, state, year, month, count):
        return f'Tier {label}: {state}/{year}/{month} ({count} records)'

    def try_pysus():
        now = datetime.now()
        sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
        for state in STATES:
            for y in range(sy, sy - 2, -1):
                for m in range(sm if y == sy else 12, 0, -1):
                    try:
                        df = sih(state=state, year=y, month=[m], group='RD', as_dataframe=True)
                        if df is not None and len(df) > 0 and 'MORTE' in df.columns:
                            print(fmt_tier('1: SIH-RD', state, y, m, len(df)))
                            return pl.from_pandas(df)
                    except Exception:
                        continue
            for y in range(sy, sy - 2, -1):
                try:
                    df = sim(state=state, year=y, as_dataframe=True)
                    if df is not None and len(df) > 0:
                        print(fmt_tier('2: SIM', state, y, 'all', len(df)))
                        df['MORTE'] = 1
                        return pl.from_pandas(df)
                except Exception:
                    continue
        for y in range(sy, sy - 2, -1):
            try:
                df = sinan(disease='deng', year=y, as_dataframe=True)
                if df is not None and len(df) > 0:
                    print(fmt_tier('3: SINAN-deng', 'BR', y, 'all', len(df)))
                    if 'MORTE' not in df.columns:
                        df['MORTE'] = 0
                    return pl.from_pandas(df)
            except Exception:
                continue
        return None

    def try_direct_ftp():
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except ImportError:
            return None
        from pysus.api.extensions import DBC
        now = datetime.now()
        sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
        base = 'ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/199301_/Dados/'
        for state in ['SP', 'RJ', 'MG']:
            for y in range(sy, sy - 2, -1):
                for m in range(sm if y == sy else 12, 0, -1):
                    fname = f'RD{state}{y % 100:02d}{m:02d}.dbc'
                    try:
                        with urllib.request.urlopen(base + fname, timeout=30) as r:
                            dbc_path = Path('/tmp') / fname
                            dbc_path.write_bytes(r.read())
                        df = asyncio.run(DBC(path=dbc_path).load())
                        if df is not None and len(df) > 0:
                            print(fmt_tier('4: FTP direct', state, y, m, len(df)))
                            return pl.from_pandas(df)
                    except Exception:
                        continue
        return None

    def generate_sample():
        print('Tier 5: Generated demo sample (real data unavailable)')
        print('NOTE: This is not real data — results are for pipeline demo only.')
        np.random.seed(42)
        n = 1000
        return pl.DataFrame({
            'MORTE': np.random.choice([0, 1], n, p=[0.92, 0.08]).astype(np.int64),
            'IDADE': np.random.randint(0, 100, n).astype(np.float64),
            'SEXO': np.random.choice(['0', '1'], n),
            'DIAS_PERMANENCIA': np.clip(np.random.poisson(5, n), 0, 100).astype(np.float64),
            'VALOR_TOTAL': np.round(np.random.lognormal(7, 1.5, n), 2),
            'PROCEDIMENTO_PRINCIPAL': np.random.choice(['AB', 'CD', 'EF', 'GH', 'IJ'], n),
            'MUNICIPIO_MOV': np.random.choice(['3550308', '3304557', '3106200'], n),
        })

    df = try_pysus()
    if df is None:
        df = try_direct_ftp()
    if df is None:
        df = generate_sample()

    df = df.with_columns(
        pl.col('MORTE').cast(pl.Int64).fill_null(0)
    )

    cols_to_keep = [
        c for c in df.columns
        if not c.startswith(('N_AIH', 'IDENT', 'CNS', 'CPF', 'CNPJ'))
    ]
    df = df.select(
        ['MORTE'] + [c for c in cols_to_keep if c != 'MORTE']
    ).head(5000)
    df.write_parquet(f'{WORKING_DIR}/raw_sample.parquet')
    
    !datalus ingest {WORKING_DIR}/raw_sample.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE
    !datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 1 --batch-size 256
else:
    print('Verified: Training artifacts are ready.')


## 2. Ab-Initio Generation & Distribution Analysis (KDE)
Generating a large synthetic dataset and comparing its statistical distribution to the real data using Kernel Density Estimation (KDE).

In [ ]:
!datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 10000 --cfg-scale 1.0

import matplotlib.pyplot as plt
import seaborn as sns

real = pl.read_parquet(f'{WORKING_DIR}/processed.parquet')
synth = pl.read_parquet(f'{WORKING_DIR}/synthetic.parquet')

plt.figure(figsize=(12, 6))
sns.kdeplot(data=real['IDADE'], label='Real Data', fill=True, alpha=0.3)
sns.kdeplot(data=synth['IDADE'], label='Synthetic Data', fill=True, alpha=0.3)
plt.title('Kernel Density Estimation: Real vs Synthetic (Age / IDADE)')
plt.xlabel('Age (Years)')
plt.ylabel('Density')
plt.legend()
plt.show()

## 3. Advanced Data Augmentation
Demonstrating how synthetic augmentation prevents overfitting in downstream machine learning tasks.

In [ ]:
seed = synth.sample(n=1000)
seed.write_parquet(f'{WORKING_DIR}/seed_augment.parquet')

!datalus augment {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/seed_augment.parquet {WORKING_DIR}/augmented.parquet --n-records 1000

## 4. Algorithmic Fairness: Minority Class Balancing
Addressing class imbalance (e.g., rare diseases or specific demographics) through oversampling without row duplication.

In [ ]:
!datalus balance {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/balanced.parquet MORTE '{"1": 5000}'

balanced = pl.read_parquet(f'{WORKING_DIR}/balanced.parquet')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
real['MORTE'].value_counts().to_pandas().plot(kind='bar', x='MORTE', y='count', ax=ax1, color='salmon')
ax1.set_title('Original Class Imbalance (Real)')
balanced['MORTE'].value_counts().to_pandas().plot(kind='bar', x='MORTE', y='count', ax=ax2, color='skyblue')
ax2.set_title('Balanced Class Distribution (Synthetic)')
plt.show()

## 5. Tabular Inpainting (Imputation)
Using RePaint-style inference to conditionally fill null values while preserving observed fields.

In [ ]:
mask = np.random.rand(100) > 0.5
incomplete = real.head(100).with_columns(
    pl.when(pl.lit(mask)).then(None).otherwise(pl.col('IDADE')).alias('IDADE')
)
incomplete.write_parquet(f'{WORKING_DIR}/incomplete.parquet')

!datalus inpaint {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/incomplete.parquet {WORKING_DIR}/inpainted.parquet

## 6. Counterfactual Interventions & Policy Simulation
Simulating "What if" scenarios by modifying patient traits and allowing the model to regenerate the clinical record coherently.

In [ ]:
!datalus counterfactual {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/counterfactual.parquet '{"SEXO": "1"}'